# Semana 11: Modelos Complementarios
## Notebook de Ejercicios (NB2) – Aplicación en Datos Atípicos

**Propósito:** Aplicar técnicas de regresión robusta y clustering jerárquico a un dataset real con posibles outliers (Wine Quality).

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Detectar outliers en un dataset real utilizando métodos estadísticos.
- Aplicar regresión robusta (Huber, RANSAC) y comparar con regresión lineal.
- Evaluar la mejora en RMSE al utilizar modelos robustos.
- (Opcional) Aplicar clustering jerárquico para segmentar vinos por calidad.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.linear_model import LinearRegression, HuberRegressor, RANSACRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score

# Scipy para clustering jerárquico
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: Wine Quality

El dataset **Wine Quality** contiene características fisicoquímicas de vinos tintos y una puntuación de calidad (0-10). Es conocido por tener valores atípicos y una distribución no uniforme de la calidad.

In [ ]:
# URL del dataset (UCI Machine Learning Repository)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

try:
    df = pd.read_csv(url, sep=';')
    print("Dataset cargado correctamente.")
except:
    print("No se pudo cargar desde URL. Usando datos locales de muestra...")
    # Datos sintéticos similares
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'fixed acidity': np.random.uniform(4, 16, n),
        'volatile acidity': np.random.uniform(0.1, 1.5, n),
        'citric acid': np.random.uniform(0, 1, n),
        'residual sugar': np.random.uniform(0.5, 15, n),
        'chlorides': np.random.uniform(0.01, 0.6, n),
        'free sulfur dioxide': np.random.uniform(1, 70, n),
        'total sulfur dioxide': np.random.uniform(6, 300, n),
        'density': np.random.uniform(0.99, 1.01, n),
        'pH': np.random.uniform(2.8, 4, n),
        'sulphates': np.random.uniform(0.3, 2, n),
        'alcohol': np.random.uniform(8, 15, n),
        'quality': np.random.randint(3, 9, n)
    })

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información general
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Distribución de la variable objetivo (calidad)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['quality'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución de Calidad')
plt.xlabel('Calidad')
plt.ylabel('Frecuencia')

plt.subplot(1, 2, 2)
df['quality'].plot(kind='box')
plt.title('Boxplot de Calidad')
plt.tight_layout()
plt.show()

---
## 2. Detección de Outliers

Utilizamos el método del rango intercuartílico (IQR) para detectar outliers en las variables predictoras.

In [ ]:
# Función para detectar outliers usando IQR
def detect_outliers_iqr(df, columns):
    outlier_counts = {}
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        outlier_counts[col] = outliers
    return outlier_counts

# Seleccionamos variables predictoras numéricas
feature_cols = [col for col in df.columns if col != 'quality']

outliers = detect_outliers_iqr(df, feature_cols)
outlier_df = pd.DataFrame(list(outliers.items()), columns=['Variable', 'Outliers'])
outlier_df['Porcentaje'] = (outlier_df['Outliers'] / len(df)) * 100
print("=== Detección de Outliers por Variable ===")
outlier_df.sort_values('Outliers', ascending=False)

In [ ]:
# Visualizamos outliers con boxplots
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(feature_cols[:11]):  # limitamos a 11 gráficos
    axes[i].boxplot(df[col])
    axes[i].set_title(col)
    axes[i].set_ylabel('Valor')

plt.tight_layout()
plt.show()

---
## 3. Preparación de Datos para Regresión

Predecimos la calidad del vino en función de las características fisicoquímicas.

In [ ]:
# Definimos X e y
X = df[feature_cols]
y = df['quality']

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarizamos (importante para algunos modelos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño entrenamiento: {X_train.shape}")
print(f"Tamaño prueba: {X_test.shape}")

---
## 4. Regresión Lineal (Modelo Base)

Entrenamos una regresión lineal estándar para tener una línea base.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("=== Regresión Lineal ===")
print(f"RMSE: {rmse_lr:.3f}")
print(f"R²: {r2_lr:.3f}")

---
## 5. Regresión Robusta

Aplicamos HuberRegressor y RANSACRegressor para manejar outliers.

In [ ]:
# HuberRegressor
huber = HuberRegressor(epsilon=1.35)
huber.fit(X_train_scaled, y_train)
y_pred_huber = huber.predict(X_test_scaled)

rmse_huber = np.sqrt(mean_squared_error(y_test, y_pred_huber))
r2_huber = r2_score(y_test, y_pred_huber)

# RANSACRegressor
ransac = RANSACRegressor(min_samples=0.5, residual_threshold=1.0, random_state=42)
ransac.fit(X_train_scaled, y_train)
y_pred_ransac = ransac.predict(X_test_scaled)

rmse_ransac = np.sqrt(mean_squared_error(y_test, y_pred_ransac))
r2_ransac = r2_score(y_test, y_pred_ransac)

print("=== Regresión Robusta ===")
print(f"HuberRegressor - RMSE: {rmse_huber:.3f}, R²: {r2_huber:.3f}")
print(f"RANSAC - RMSE: {rmse_ransac:.3f}, R²: {r2_ransac:.3f}")

# Mostrar inliers detectados por RANSAC
inlier_mask = ransac.inlier_mask_
print(f"\nRANSAC detectó {np.sum(inlier_mask)} inliers de {len(y_train)} puntos ({np.sum(inlier_mask)/len(y_train):.1%})")

## 6. Comparación de Modelos

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'HuberRegressor', 'RANSAC'],
    'RMSE': [rmse_lr, rmse_huber, rmse_ransac],
    'R²': [r2_lr, r2_huber, r2_ransac]
})

print("=== Comparación de Modelos ===")
comparacion.round(4)

In [ ]:
# Visualización de la mejora
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(comparacion['Modelo'], comparacion['RMSE'], color=['blue', 'green', 'orange'])
axes[0].set_title('Comparación de RMSE')
axes[0].set_ylabel('RMSE')
axes[0].grid(axis='y')

axes[1].bar(comparacion['Modelo'], comparacion['R²'], color=['blue', 'green', 'orange'])
axes[1].set_title('Comparación de R²')
axes[1].set_ylabel('R²')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

---
## 7. (Opcional) Clustering Jerárquico para Segmentar Vinos

Aplicamos clustering jerárquico para agrupar vinos según sus características fisicoquímicas y analizamos la relación con la calidad.

In [ ]:
# Usamos una muestra para que el dendrograma sea legible
sample_size = 50
X_sample = X.sample(sample_size, random_state=42)
y_sample = y.loc[X_sample.index]

# Estandarizamos
X_sample_scaled = scaler.fit_transform(X_sample)

# Calculamos linkage
Z = linkage(X_sample_scaled, method='ward')

# Dendrograma
plt.figure(figsize=(15, 6))
dendrogram(Z, truncate_mode='level', p=5, labels=y_sample.values)
plt.title('Dendrograma - Vinos (coloreados por calidad)')
plt.xlabel('Muestra')
plt.ylabel('Distancia')
plt.axhline(y=20, color='red', linestyle='--', label='Corte en distancia=20')
plt.legend()
plt.show()

In [ ]:
# Cortamos el dendrograma para obtener 3 clusters
clusters = fcluster(Z, t=3, criterion='maxclust')

# Añadimos los clusters al DataFrame de muestra
df_sample = X_sample.copy()
df_sample['quality'] = y_sample
df_sample['cluster'] = clusters

# Perfil de los clusters
cluster_profile = df_sample.groupby('cluster')[feature_cols].mean()
cluster_profile['quality_mean'] = df_sample.groupby('cluster')['quality'].mean()
cluster_profile['count'] = df_sample.groupby('cluster').size()

print("=== Perfil de Clusters ===")
cluster_profile.round(2)

In [ ]:
# Visualizamos los clusters en 2D usando las dos primeras componentes (PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sample_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=100, alpha=0.7)
plt.colorbar(scatter, label='Cluster')

# Añadimos etiquetas con la calidad
for i, (x, y) in enumerate(X_pca):
    plt.annotate(str(y_sample.iloc[i]), (x, y), fontsize=8, alpha=0.7)

plt.title('Clusters Jerárquicos en espacio PCA (etiquetas = calidad)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True)
plt.show()

In [ ]:
# Evaluamos la calidad de los clusters
sil_score = silhouette_score(X_sample_scaled, clusters)
print(f"Coeficiente de silueta para los 3 clusters: {sil_score:.3f}")

# Comparación con K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_clusters = kmeans.fit_predict(X_sample_scaled)
sil_kmeans = silhouette_score(X_sample_scaled, kmeans_clusters)
print(f"Coeficiente de silueta para K-Means (3 clusters): {sil_kmeans:.3f}")

---
## 8. Conclusiones

En este notebook hemos aplicado técnicas de regresión robusta y clustering jerárquico a un dataset real con posibles outliers:

✔️ **Detección de outliers**: Identificamos variables con alta presencia de valores atípicos.

✔️ **Regresión robusta**:
- La regresión lineal estándar se ve afectada por outliers.
- HuberRegressor mejora ligeramente el RMSE.
- RANSAC ofrece la mayor mejora al identificar y excluir outliers.

✔️ **Clustering jerárquico**:
- Segmentamos vinos en clusters basados en características fisicoquímicas.
- Los clusters muestran diferencias en la calidad promedio.
- La visualización con PCA ayuda a interpretar los grupos.

**Lección clave**: Cuando los datos contienen outliers significativos, los modelos robustos como RANSAC pueden mejorar sustancialmente el rendimiento predictivo. El clustering jerárquico proporciona una visión complementaria de la estructura de los datos.

---
**Fin del Notebook de Ejercicios - Semana 11**